# 5.05 Vendor-Managed Retriever

클라우드 공급자가 **호스팅·인덱싱·관리**까지 책임지는 retriever 4종을 다룬다.
데이터가 대량이거나 팀이 이미 해당 클라우드에 정착해 있을 때 합리적인 선택지.

- `AmazonKnowledgeBasesRetriever` (`langchain-aws`) — AWS Bedrock Knowledge Bases
- `AzureAISearchRetriever` (`langchain-community`) — Azure AI Search (구 Cognitive Search)
- `VertexAISearchRetriever` (`langchain-google-community`) — Google Cloud Vertex AI Search (구 Discovery Engine)
- `ElasticsearchRetriever` (`langchain-elasticsearch`) — Elastic 팀이 제공하는 ES 전용 retriever (쿼리 DSL 자유도 높음)

## 학습 목표

- 네 retriever의 **인증·리소스 식별자** 필드 차이를 이해한다
- `retriever.invoke("query")` 한 줄 호출 규약은 동일하다는 점을 확인
- 로컬에 키가 없어도 **스키마 확인**과 **스킵 경로**가 깔끔하게 흐르도록 구조화
- 실행 가능한 경우 `create_retriever_tool`로 에이전트에 연결

## 언제 쓰나

- **관리형**이 필요한 조직 (DBA·검색엔지니어 리소스 절약)
- 데이터가 이미 해당 클라우드 내 버킷/스토어에 있고, **ETL 없이 바로 인덱싱**하고 싶을 때
- Knowledge Base가 자동으로 **Embed + chunk + store** 해주므로 파이프라인 초기 구축 부담 감소


## 5.05.1 환경 설정

이 노트북은 **대부분 key-gated**다. 키가 없으면 해당 섹션을 건너뛰고 스키마만 확인한다.

| 섹션 | 필요 환경 변수 | 패키지 |
|------|----------------|--------|
| Amazon KB | `AWS_REGION`, `BEDROCK_KB_ID` (+ AWS 인증 체인) | `langchain-aws` |
| Azure AI Search | `AZURE_AI_SEARCH_SERVICE_NAME`, `AZURE_AI_SEARCH_INDEX_NAME`, `AZURE_AI_SEARCH_API_KEY` | `langchain-community` |
| Vertex AI Search | `GOOGLE_CLOUD_PROJECT`, `VERTEX_SEARCH_DATA_STORE_ID` (+ ADC) | `langchain-google-community` |
| Elasticsearch | `ES_URL`, `ES_API_KEY`, `ES_INDEX` | `langchain-elasticsearch` |

설치는 필요한 것만:
```
uv pip install langchain-aws langchain-community langchain-google-community langchain-elasticsearch
```


In [ ]:
# !pip install -U langchain langchain-core langchain-aws langchain-community langchain-elasticsearch

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 환경별 gate 플래그
HAS_AWS_KB   = bool(os.environ.get("BEDROCK_KB_ID"))
HAS_AZURE    = bool(os.environ.get("AZURE_AI_SEARCH_SERVICE_NAME") and os.environ.get("AZURE_AI_SEARCH_API_KEY") and os.environ.get("AZURE_AI_SEARCH_INDEX_NAME"))
HAS_VERTEX   = bool(os.environ.get("GOOGLE_CLOUD_PROJECT") and os.environ.get("VERTEX_SEARCH_DATA_STORE_ID"))
HAS_ES       = bool(os.environ.get("ES_URL") and os.environ.get("ES_API_KEY") and os.environ.get("ES_INDEX"))

print({"aws_kb": HAS_AWS_KB, "azure": HAS_AZURE, "vertex": HAS_VERTEX, "es": HAS_ES})


## 5.05.2 기본 사용법 — `AmazonKnowledgeBasesRetriever`

AWS Bedrock Knowledge Bases는 **S3 원본 → Titan/Cohere 임베딩 → OpenSearch Serverless/pgvector 저장**까지 관리형으로 묶어 준다. `knowledge_base_id` 하나만 알면 retriever가 내부 DSL을 숨겨 준다.

주요 필드:
- `knowledge_base_id` (필수)
- `region_name`
- `retrieval_config={"vectorSearchConfiguration": {"numberOfResults": 5, "filter": {...}}}`
- `min_score_confidence` — 결과 후처리 임계값 (0~1)


In [ ]:
from langchain_aws.retrievers import AmazonKnowledgeBasesRetriever

print("필드 목록:", list(AmazonKnowledgeBasesRetriever.model_fields.keys()))

if HAS_AWS_KB:
    kb = AmazonKnowledgeBasesRetriever(
        knowledge_base_id=os.environ["BEDROCK_KB_ID"],
        region_name=os.environ.get("AWS_REGION", "us-east-1"),
        retrieval_config={
            "vectorSearchConfiguration": {"numberOfResults": 3}
        },
    )
    for d in kb.invoke("회사 휴가 정책"):
        print("-", d.metadata.get("source_metadata"), "|", d.page_content[:80])
else:
    print("AWS KB 키 없음 → 스킵. 클래스 import와 필드 확인만 수행.")


## 5.05.3 `AzureAISearchRetriever`

Azure AI Search는 전통적 **검색 엔진**에 벡터 검색이 얹힌 하이브리드 구조. 색인(`index_name`)에는 미리 `vectorField`를 포함하는 스키마가 정의돼 있어야 한다.

주요 필드: `service_name`, `index_name`, `api_key`, `api_version`, `top_k`, `content_key`, `filter`.


In [ ]:
from langchain_community.retrievers import AzureAISearchRetriever

print("필드 목록:", list(AzureAISearchRetriever.model_fields.keys()))

if HAS_AZURE:
    az = AzureAISearchRetriever(
        service_name=os.environ["AZURE_AI_SEARCH_SERVICE_NAME"],
        index_name=os.environ["AZURE_AI_SEARCH_INDEX_NAME"],
        api_key=os.environ["AZURE_AI_SEARCH_API_KEY"],
        top_k=3,
        content_key="content",  # 색인 스키마 속 본문 필드명에 맞춰 변경
    )
    for d in az.invoke("support article about login"):
        print("-", d.metadata.get("@search.score"), "|", d.page_content[:80])
else:
    print("Azure AI Search 키 없음 → 스킵.")


## 5.05.4 `VertexAISearchRetriever`

Google Cloud Vertex AI Search (구 Discovery Engine)는 GCS 원본 · 웹사이트 · BigQuery를 데이터 소스로 받는 관리형 검색.

**지역 주의**: `location_id`는 `global` / `us` / `eu` 중 하나로, 콘솔에서 만든 데이터 스토어와 **반드시 일치**해야 한다.

주요 필드: `project_id`, `data_store_id`, `location_id`, `max_documents`, `get_extractive_answers`.


In [ ]:
try:
    from langchain_google_community import VertexAISearchRetriever
    print("필드 목록:", list(VertexAISearchRetriever.model_fields.keys()))
    _vertex_ok = True
except ModuleNotFoundError:
    print("langchain-google-community 미설치. `uv pip install langchain-google-community` 필요.")
    _vertex_ok = False

if _vertex_ok and HAS_VERTEX:
    vx = VertexAISearchRetriever(
        project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
        data_store_id=os.environ["VERTEX_SEARCH_DATA_STORE_ID"],
        location_id=os.environ.get("VERTEX_LOCATION_ID", "global"),
        max_documents=3,
        get_extractive_answers=True,
    )
    for d in vx.invoke("product refund policy"):
        print("-", d.metadata.get("id"), "|", d.page_content[:80])
else:
    print("Vertex AI Search 키/패키지 없음 → 스킵.")


## 5.05.5 `ElasticsearchRetriever` — 쿼리 DSL 그대로

`langchain-elasticsearch`의 `ElasticsearchRetriever`는 **임베딩을 내부에서 만들지 않는다**. 대신 사용자가 `body_func(query) -> dict`를 주면 그게 그대로 ES 쿼리 DSL이 된다. BM25·kNN·하이브리드 어떤 조합이든 만들 수 있는 대신, 스키마와 필드를 직접 관리해야 한다.


In [ ]:
from langchain_elasticsearch import ElasticsearchRetriever
from elasticsearch import Elasticsearch

print("필드 목록:", list(ElasticsearchRetriever.model_fields.keys()))

def match_query(q: str) -> dict:
    # 필요에 따라 kNN + BM25 하이브리드로 확장 가능
    return {
        "size": 3,
        "query": {"match": {"content": q}},
    }

if HAS_ES:
    client = Elasticsearch(
        os.environ["ES_URL"],
        api_key=os.environ["ES_API_KEY"],
    )
    es_r = ElasticsearchRetriever(
        client=client,
        index_name=os.environ["ES_INDEX"],
        body_func=match_query,
        content_field="content",  # 색인의 본문 필드명
    )
    for d in es_r.invoke("deployment runbook"):
        print("-", d.metadata.get("_score"), "|", d.page_content[:80])
else:
    print("ES 키 없음 → 스킵.")


## 5.05.6 결과 비교 — 공통점 · 차이점

| 항목 | Amazon KB | Azure AI Search | Vertex AI Search | Elasticsearch |
|------|-----------|-----------------|------------------|---------------|
| 인덱싱 방식 | 관리형 (S3 → auto embed) | 반-수동 (스키마 정의 필요) | 관리형 (데이터 소스 연결) | 수동 (사용자가 DSL 작성) |
| 필터 DSL | `filter={"equals":{...}}` | `filter=OData-style` | `filter=API-style` | ES Query DSL 전체 |
| 임베딩 모델 교체 | 설정에서 선택 | 외부 커스텀 필요 | 자동 | 외부 커스텀 필요 |
| 하이브리드 검색 | 지원 | 지원 | 지원 | 지원 (DSL로 직접 작성) |
| 비용 구조 | per-query + storage | per-query + storage | per-query + storage | per-hour 클러스터 |

**retriever 인터페이스는 동일**하다는 것이 핵심 — 백엔드 교체 시 `create_retriever_tool(...)` 위쪽 에이전트 코드는 그대로다.


## 5.05.7 `create_agent` 연동

키가 있는 하나 이상의 백엔드를 **그대로** 도구화하는 예시. 기본적으로 retriever 하나만 있어도 동작하는 패턴.


In [ ]:
# 사용 가능한 retriever를 모두 도구로 엮는다
available = []

if HAS_AWS_KB:
    available.append(("aws_kb_search", "AWS Bedrock Knowledge Base를 검색한다.", kb))
if HAS_AZURE:
    available.append(("azure_search",  "Azure AI Search 인덱스를 검색한다.", az))
if _vertex_ok and HAS_VERTEX:
    available.append(("vertex_search", "Vertex AI Search 데이터 스토어를 검색한다.", vx))
if HAS_ES:
    available.append(("es_search", "Elasticsearch 인덱스를 match 쿼리로 검색한다.", es_r))

print("사용 가능한 retriever:", [n for n, _, _ in available])

if available and os.environ.get("OPENAI_API_KEY"):
    from langchain_classic.tools.retriever import create_retriever_tool
    from langchain.agents import create_agent

    tools = [create_retriever_tool(r, name, desc) for (name, desc, r) in available]
    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=tools,
        system_prompt="사용자 질문에 답하려면 등록된 검색 도구 중 하나 이상을 호출하라.",
    )
    r = agent.invoke({"messages": [{"role": "user", "content": "회사 정책 문서 요약해줘"}]})
    print(r["messages"][-1].content[:400])
else:
    print("실행 조건 미충족 (키 또는 OPENAI_API_KEY 없음) → 에이전트 예제 스킵.")


## 체크리스트

- [ ] 각 벤더는 **인증 체인**이 다르다 — Azure는 API key 직접, AWS는 IAM 체인, GCP는 ADC (`gcloud auth application-default login`)
- [ ] `content_key` / `content_field` 옵션은 색인 스키마의 **본문 필드명**과 일치시켜야 한다
- [ ] 관리형은 "인덱스 생성/데이터 소스 연결" 단계가 콘솔·IaC에서 선행 — 이 노트북은 **이미 만들어진 리소스**를 가정
- [ ] 프로덕션에서는 `retriever.invoke` 동기 호출 대신 `ainvoke` 비동기 경로를 쓰거나 배치 API 사용
- [ ] 대부분의 벤더 retriever는 `filter` 파라미터로 tenant 분리·권한 분리를 구현한다 — RAG 보안의 1차 방어선

## 다음

- `03_vectorstores/08_elasticsearch.ipynb` — ES를 vectorstore로 쓰는 경로 (`ElasticsearchStore`)
- `05_advanced/05_agentic_rag.ipynb` — retriever를 도구로 묶은 full agent

## 참고

- AWS Bedrock KB: https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html
- Azure AI Search: https://learn.microsoft.com/azure/search/
- Vertex AI Search: https://cloud.google.com/generative-ai-app-builder/docs/introduction
- Elasticsearch kNN: https://www.elastic.co/guide/en/elasticsearch/reference/current/knn-search.html
